# Week 4: Streaming Computation Across Tiles

This notebook extends Week 3 by implementing streaming computation across tiles for attention workloads. The goal is to improve execution flow for scenarios such as autoregressive generation and streaming inference, where tokens arrive incrementally and attention must be computed over growing prefixes rather than full fixed sequences.

In earlier weeks, this project established a baseline attention implementation, defined a Python-embedded DSL for scenario-aware attention specification, and introduced a tiled attention transformation inspired by FlashAttention-style execution. This week adds a streaming tiled execution path that combines two ideas: block-wise computation and incremental prefix-based attention.

The goals of this notebook are:

1. Reuse the baseline attention, streaming attention, tiled attention, and DSL interface from earlier weeks.
2. Implement streaming tiled attention.
3. Improve numerical stability during tiled softmax computation.
4. Extend DSL lowering to support scenario-aware routing to streaming tiled execution.
5. Verify correctness against reference implementations.
6. Compare runtime and approximate memory behavior across execution strategies.

This implementation remains a high-level Python/PyTorch prototype rather than a fused kernel, but it moves the project closer to customized attention execution for real deployment scenarios.

## Week 4 Objectives

This week focuses on combining tiled execution with streaming-style incremental attention. In a streaming inference setting, each new query token only needs to attend to previously seen tokens. This changes the execution pattern and creates opportunities for more specialized attention implementations.

The purpose of this notebook is therefore to demonstrate that attention can be computed incrementally across tiles, while preserving correctness and improving the conceptual alignment between execution strategy and application scenario.

In [ ]:
import math
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


## Baseline Attention Functions

The following functions are reused from earlier weeks and serve as reference implementations for correctness checks.

In [ ]:
def baseline_attention(Q, K, V, causal=False):
    """
    Standard scaled dot-product attention.
    Supports optional causal masking.
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if causal:
        seq_len_q = Q.size(-2)
        seq_len_k = K.size(-2)
        mask = torch.triu(
            torch.ones(seq_len_q, seq_len_k, device=Q.device, dtype=torch.bool),
            diagonal=1
        )
        scores = scores.masked_fill(mask, float("-inf"))

    weights = torch.softmax(scores, dim=-1)
    return torch.matmul(weights, V)

In [ ]:
def streaming_attention(Q_seq, K_seq, V_seq):
    """
    Simple streaming-style reference attention.
    At time step t, attention is computed only over the prefix up to t.
    """
    outputs = []

    for t in range(Q_seq.size(1)):
        Q_t = Q_seq[:, t:t+1, :]
        K_t = K_seq[:, :t+1, :]
        V_t = V_seq[:, :t+1, :]
        out_t = baseline_attention(Q_t, K_t, V_t, causal=False)
        outputs.append(out_t)

    return torch.cat(outputs, dim=1)

In [ ]:
def tiled_attention(Q, K, V, tile_size=64, causal=False):
    """
    High-level tiled attention implementation.
    """
    B, N, D = Q.shape
    output = torch.zeros_like(Q)

    for i in range(0, N, tile_size):
        i_end = min(i + tile_size, N)
        Q_tile = Q[:, i:i_end, :]

        score_tiles = []
        value_tiles = []

        for j in range(0, N, tile_size):
            j_end = min(j + tile_size, N)
            K_tile = K[:, j:j_end, :]
            V_tile = V[:, j:j_end, :]

            scores = torch.matmul(Q_tile, K_tile.transpose(-2, -1)) / math.sqrt(D)

            if causal:
                q_idx = torch.arange(i, i_end, device=Q.device).unsqueeze(1)
                k_idx = torch.arange(j, j_end, device=Q.device).unsqueeze(0)
                causal_mask = k_idx > q_idx
                scores = scores.masked_fill(causal_mask.unsqueeze(0), float("-inf"))

            score_tiles.append(scores)
            value_tiles.append(V_tile)

        full_tile_scores = torch.cat(score_tiles, dim=-1)
        weights = torch.softmax(full_tile_scores, dim=-1)

        tile_output = torch.zeros(B, i_end - i, D, device=Q.device, dtype=Q.dtype)

        start = 0
        for V_tile in value_tiles:
            end = start + V_tile.size(1)
            tile_weights = weights[:, :, start:end]
            tile_output += torch.matmul(tile_weights, V_tile)
            start = end

        output[:, i:i_end, :] = tile_output

    return output

## DSL Interface

The DSL from Week 2 is reused again here. In Week 4, the fields `streaming` and `tile_size` now jointly determine the execution path during lowering.

In [ ]:
class AttentionSpec:
    """
    Lightweight DSL object for declarative attention specification.
    """
    def __init__(
        self,
        Q,
        K,
        V,
        name="attention_op",
        scenario="generic",
        causal=False,
        tile_size=None,
        streaming=False,
        variable_length=False,
        cache_kv=False
    ):
        self.Q = Q
        self.K = K
        self.V = V
        self.name = name
        self.scenario = scenario
        self.causal = causal
        self.tile_size = tile_size
        self.streaming = streaming
        self.variable_length = variable_length
        self.cache_kv = cache_kv

    def summary(self):
        return {
            "name": self.name,
            "scenario": self.scenario,
            "causal": self.causal,
            "tile_size": self.tile_size,
            "streaming": self.streaming,
            "variable_length": self.variable_length,
            "cache_kv": self.cache_kv,
            "Q_shape": tuple(self.Q.shape),
            "K_shape": tuple(self.K.shape),
            "V_shape": tuple(self.V.shape),
        }

## Numerically Stable Softmax Helper

When working with tiled or streaming computations, numerical stability becomes important. A common technique is to subtract the maximum score before applying softmax. This does not change the mathematical result, but it helps avoid overflow and improves stability.

In [ ]:
def stable_softmax(scores, dim=-1):
    """
    Numerically stable softmax using max subtraction.
    """
    max_scores = scores.max(dim=dim, keepdim=True).values
    shifted_scores = scores - max_scores
    exp_scores = torch.exp(shifted_scores)
    return exp_scores / exp_scores.sum(dim=dim, keepdim=True)

## Streaming Tiled Attention

The following function combines streaming-style prefix attention with tiled execution. At each time step, the current query attends only to the valid prefix of keys and values seen so far. That prefix is processed in smaller key-value tiles.

This is a high-level prototype intended to model the execution structure of streaming tiled attention rather than a fused production implementation.

In [ ]:
def streaming_tiled_attention(Q_seq, K_seq, V_seq, tile_size=64):
    """
    Streaming tiled attention prototype.

    For each time step t:
    - compute attention for Q_t
    - attend only to prefix K[:t+1], V[:t+1]
    - process that prefix in tiles
    - use numerically stable softmax
    """
    B, N, D = Q_seq.shape
    outputs = []

    for t in range(N):
        Q_t = Q_seq[:, t:t+1, :]
        prefix_len = t + 1

        score_tiles = []
        value_tiles = []

        for j in range(0, prefix_len, tile_size):
            j_end = min(j + tile_size, prefix_len)
            K_tile = K_seq[:, j:j_end, :]
            V_tile = V_seq[:, j:j_end, :]

            scores = torch.matmul(Q_t, K_tile.transpose(-2, -1)) / math.sqrt(D)
            score_tiles.append(scores)
            value_tiles.append(V_tile)

        full_scores = torch.cat(score_tiles, dim=-1)
        weights = stable_softmax(full_scores, dim=-1)

        out_t = torch.zeros(B, 1, D, device=Q_seq.device, dtype=Q_seq.dtype)

        start = 0
        for V_tile in value_tiles:
            end = start + V_tile.size(1)
            tile_weights = weights[:, :, start:end]
            out_t += torch.matmul(tile_weights, V_tile)
            start = end

        outputs.append(out_t)

    return torch.cat(outputs, dim=1)

## Quick Sanity Check for Streaming Tiled Attention

Before integrating streaming tiled attention into DSL lowering, we verify that it matches the simpler streaming reference implementation.

In [ ]:
B, N, D = 1, 32, 32

Q = torch.randn(B, N, D, device=device)
K = torch.randn(B, N, D, device=device)
V = torch.randn(B, N, D, device=device)

stream_ref = streaming_attention(Q, K, V)
stream_tiled = streaming_tiled_attention(Q, K, V, tile_size=8)

stream_max_diff = (stream_ref - stream_tiled).abs().max().item()
print("Maximum absolute difference:", stream_max_diff)
print("Outputs match:", torch.allclose(stream_ref, stream_tiled, atol=1e-6))

Maximum absolute difference: 1.1920928955078125e-07
Outputs match: True


## Stability Check

This small check compares standard softmax with the stable softmax helper on a sample score tensor. The purpose is to confirm that the stable implementation produces equivalent results while being safer for larger score values.

In [ ]:
sample_scores = torch.randn(2, 4, 6, device=device) * 10
softmax_standard = torch.softmax(sample_scores, dim=-1)
softmax_stable = stable_softmax(sample_scores, dim=-1)

stability_diff = (softmax_standard - softmax_stable).abs().max().item()
print("Softmax difference:", stability_diff)
print("Softmax outputs match:", torch.allclose(softmax_standard, softmax_stable, atol=1e-6))

Softmax difference: 0.0
Softmax outputs match: True


## Updated DSL Lowering

The lowering stage is now extended to support a new execution path:

1. If `streaming=True` and `tile_size` is provided, use streaming tiled attention.
2. If `streaming=True` without tiling, use the streaming reference implementation.
3. If `tile_size` is provided without streaming, use tiled attention.
4. Otherwise, use standard baseline attention.

This is the clearest sign so far that the DSL is becoming a scenario-aware execution interface rather than just a wrapper around baseline attention.

In [ ]:
def lower_attention(spec):
    """
    Lower a DSL attention specification to an executable implementation.
    """
    if spec.streaming and spec.tile_size is not None:
        return streaming_tiled_attention(
            spec.Q,
            spec.K,
            spec.V,
            tile_size=spec.tile_size
        )

    if spec.streaming:
        return streaming_attention(spec.Q, spec.K, spec.V)

    if spec.tile_size is not None:
        return tiled_attention(
            spec.Q,
            spec.K,
            spec.V,
            tile_size=spec.tile_size,
            causal=spec.causal
        )

    return baseline_attention(spec.Q, spec.K, spec.V, causal=spec.causal)

## Scenario 1: Multi-Tenant Serving

This scenario still uses tiled attention. In a multi-tenant system, requests may have variable sequence lengths and different execution constraints. The current prototype represents this through metadata and tiled lowering.

In [ ]:
Q_mt = torch.randn(1, 128, 64, device=device)
K_mt = torch.randn(1, 128, 64, device=device)
V_mt = torch.randn(1, 128, 64, device=device)

multitenant_spec = AttentionSpec(
    Q=Q_mt,
    K=K_mt,
    V=V_mt,
    name="multitenant_request",
    scenario="multi_tenant",
    causal=False,
    tile_size=32,
    streaming=False,
    variable_length=True,
    cache_kv=False
)

pd.DataFrame([multitenant_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,multitenant_request,multi_tenant,False,32,False,True,False,"(1, 128, 64)","(1, 128, 64)","(1, 128, 64)"


In [ ]:
mt_dsl_output = lower_attention(multitenant_spec)
mt_ref_output = baseline_attention(Q_mt, K_mt, V_mt, causal=False)

mt_max_diff = (mt_dsl_output - mt_ref_output).abs().max().item()
print("Multi-tenant max difference:", mt_max_diff)
print("Outputs match:", torch.allclose(mt_dsl_output, mt_ref_output, atol=1e-6))

Multi-tenant max difference: 3.2782554626464844e-07
Outputs match: True


## Scenario 2: Streaming Inference

This is the main new scenario for Week 4. The specification uses both `streaming=True` and `tile_size`, so lowering now routes execution through the streaming tiled attention path.

In [ ]:
Q_stream = torch.randn(1, 48, 64, device=device)
K_stream = torch.randn(1, 48, 64, device=device)
V_stream = torch.randn(1, 48, 64, device=device)

streaming_spec = AttentionSpec(
    Q=Q_stream,
    K=K_stream,
    V=V_stream,
    name="streaming_request",
    scenario="streaming",
    causal=False,
    tile_size=16,
    streaming=True,
    variable_length=False,
    cache_kv=True
)

pd.DataFrame([streaming_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,streaming_request,streaming,False,16,True,False,True,"(1, 48, 64)","(1, 48, 64)","(1, 48, 64)"


In [ ]:
stream_dsl_output = lower_attention(streaming_spec)
stream_ref_output = streaming_attention(Q_stream, K_stream, V_stream)

stream_case_diff = (stream_dsl_output - stream_ref_output).abs().max().item()
print("Streaming max difference:", stream_case_diff)
print("Outputs match:", torch.allclose(stream_dsl_output, stream_ref_output, atol=1e-6))

Streaming max difference: 2.384185791015625e-07
Outputs match: True


## Scenario 3: Long-Context Attention

Long-context workloads continue to use the non-streaming tiled path. This reflects the idea that different scenarios can now be lowered to different execution strategies within the same DSL framework.

In [ ]:
Q_long = torch.randn(1, 256, 64, device=device)
K_long = torch.randn(1, 256, 64, device=device)
V_long = torch.randn(1, 256, 64, device=device)

long_context_spec = AttentionSpec(
    Q=Q_long,
    K=K_long,
    V=V_long,
    name="long_context_request",
    scenario="long_context",
    causal=True,
    tile_size=64,
    streaming=False,
    variable_length=False,
    cache_kv=False
)

pd.DataFrame([long_context_spec.summary()])

,name,scenario,causal,tile_size,streaming,variable_length,cache_kv,Q_shape,K_shape,V_shape
0,long_context_request,long_context,True,64,False,False,False,"(1, 256, 64)","(1, 256, 64)","(1, 256, 64)"


In [ ]:
long_dsl_output = lower_attention(long_context_spec)
long_ref_output = baseline_attention(Q_long, K_long, V_long, causal=True)

long_max_diff = (long_dsl_output - long_ref_output).abs().max().item()
print("Long-context max difference:", long_max_diff)
print("Outputs match:", torch.allclose(long_dsl_output, long_ref_output, atol=1e-6))

Long-context max difference: 7.152557373046875e-07
Outputs match: True


## Runtime Benchmarking

The benchmark below compares three strategies:

- standard baseline attention
- tiled attention
- streaming tiled attention

The goal is not necessarily to show that the prototype is faster. Because the current implementation uses Python loops rather than fused kernels, the transformed versions may be slower in wall-clock time. The main purpose is to compare execution structure and demonstrate correctness-preserving specialization.

In [ ]:
def benchmark(fn, *args, repeats=10, **kwargs):
    times = []

    for _ in range(repeats):
        if device == "cuda":
            torch.cuda.synchronize()

        start = time.time()
        fn(*args, **kwargs)

        if device == "cuda":
            torch.cuda.synchronize()

        end = time.time()
        times.append(end - start)

    return sum(times) / len(times)

In [ ]:
Q_bench = torch.randn(1, 128, 64, device=device)
K_bench = torch.randn(1, 128, 64, device=device)
V_bench = torch.randn(1, 128, 64, device=device)

baseline_time = benchmark(
    baseline_attention,
    Q_bench, K_bench, V_bench,
    repeats=10,
    causal=True
)

tiled_time = benchmark(
    tiled_attention,
    Q_bench, K_bench, V_bench,
    repeats=10,
    tile_size=32,
    causal=True
)

streaming_tiled_time = benchmark(
    streaming_tiled_attention,
    Q_bench, K_bench, V_bench,
    repeats=10,
    tile_size=32
)

timing_df = pd.DataFrame([
    {"implementation": "baseline_attention", "avg_time_sec": baseline_time},
    {"implementation": "tiled_attention", "avg_time_sec": tiled_time},
    {"implementation": "streaming_tiled_attention", "avg_time_sec": streaming_tiled_time}
])

timing_df

,implementation,avg_time_sec
0,baseline_attention,0.000422
1,tiled_attention,0.004062
2,streaming_tiled_attention,0.052502


## Approximate Intermediate Memory Comparison

This comparison gives a rough illustration of how local score computations differ across execution strategies.

For baseline attention, the full score matrix has shape `(B, N, N)`.
For tiled attention, local score blocks are smaller.
For streaming tiled attention, each time step only attends to a prefix, and each local tile is also bounded by the tile size.

This is an approximate conceptual comparison rather than a direct profiler measurement.

In [ ]:
def tensor_memory_mb(shape, dtype=torch.float32):
    bytes_per_element = torch.tensor([], dtype=dtype).element_size()
    num_elements = 1
    for dim in shape:
        num_elements *= dim
    return (num_elements * bytes_per_element) / (1024 ** 2)

B, N, tile_size = 1, 128, 32

baseline_score_mb = tensor_memory_mb((B, N, N))
tiled_local_score_mb = tensor_memory_mb((B, tile_size, tile_size))
streaming_single_step_score_mb = tensor_memory_mb((B, 1, tile_size))

memory_df = pd.DataFrame([
    {
        "tensor_type": "baseline full score matrix",
        "shape": (B, N, N),
        "approx_memory_mb": baseline_score_mb
    },
    {
        "tensor_type": "tiled local score block",
        "shape": (B, tile_size, tile_size),
        "approx_memory_mb": tiled_local_score_mb
    },
    {
        "tensor_type": "streaming tiled single-step score block",
        "shape": (B, 1, tile_size),
        "approx_memory_mb": streaming_single_step_score_mb
    }
])

memory_df

,tensor_type,shape,approx_memory_mb
0,baseline full score matrix,"(1, 128, 128)",0.062500
1,tiled local score block,"(1, 32, 32)",0.003906
2,streaming tiled single-step score block,"(1, 1, 32)",0.000122


## Scenario Summary Table

The following table summarizes the correctness results for the main Week 4 scenarios.

In [ ]:
results_df = pd.DataFrame([
    {
        "scenario": "multi_tenant",
        "max_abs_diff": mt_max_diff,
        "matches_reference": torch.allclose(mt_dsl_output, mt_ref_output, atol=1e-6)
    },
    {
        "scenario": "streaming",
        "max_abs_diff": stream_case_diff,
        "matches_reference": torch.allclose(stream_dsl_output, stream_ref_output, atol=1e-6)
    },
    {
        "scenario": "long_context",
        "max_abs_diff": long_max_diff,
        "matches_reference": torch.allclose(long_dsl_output, long_ref_output, atol=1e-6)
    }
])

results_df

,scenario,max_abs_diff,matches_reference
0,multi_tenant,3.278255e-07,True
1,streaming,2.384186e-07,True
2,long_context,7.152557e-07,True


## Week 4 Observations

This week extended the project from static tiled execution to streaming computation across tiles. The new streaming tiled attention path combines incremental prefix-based attention with block-wise processing, making it more relevant to streaming inference and autoregressive generation workloads. While the current implementation is still a high-level Python/PyTorch prototype, it demonstrates how different application scenarios can be mapped to different attention execution strategies within a unified DSL framework.

The correctness checks show that the streaming tiled implementation matches the simpler streaming reference implementation, while the long-context and multi-tenant cases continue to match their baseline references. This demonstrates that the DSL can now support multiple specialized execution paths while preserving the semantics of standard attention.

The addition of numerically stable softmax improves the reliability of tiled and streaming computations, especially when attention scores become large. Although the current prototype is still implemented with Python loops and may not outperform baseline attention in runtime, it better reflects the execution structure needed for memory-aware and scenario-specific attention optimization.

Overall, this week strengthens the project’s response to the original feedback by showing that different application scenarios can be mapped to different attention execution strategies within a unified DSL framework.

## Week 4 Deliverables Completed

- Reused the earlier baseline, streaming, tiled, and DSL components
- Implemented streaming tiled attention
- Added a numerically stable softmax helper
- Extended DSL lowering to support scenario-aware execution routing
- Verified correctness against reference implementations
- Compared runtime and approximate intermediate memory behavior
- Strengthened the project’s scenario-specific optimization story